In [0]:
from pyspark.sql.functions import (
    col, 
    sum as _sum, 
    count as _count, 
    max as _max, 
    when, 
    coalesce, 
    lit, 
    current_timestamp
)
from pyspark.sql.types import DecimalType

CATALOG_NAME = "banking_lakehouse_db2"
SCHEMA_NAME = "gold"
TABLE_NAME = "account_summary"
FULL_TABLE_NAME = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.{TABLE_NAME}"
GOLD_PATH = f"abfss://gold@bankingdelakevishal.dfs.core.windows.net/{TABLE_NAME}/"


In [0]:
# ============================================
# GOLD: ACCOUNT SUMMARY
# ============================================

# 1. Read Silver Tables
dim_account = spark.table(f"`{CATALOG_NAME}`.silver.account")
dim_customer = spark.table(f"`{CATALOG_NAME}`.silver.customer")
dim_branch = spark.table(f"`{CATALOG_NAME}`.silver.branch")
fact_tx = spark.table(f"`{CATALOG_NAME}`.silver.transaction")

# 2. Aggregate Transactions by Account
account_tx_agg = (
    fact_tx
    .groupBy("account_id")
    .agg(
        _count("transaction_id").alias("total_transactions"),
        _sum(when(col("transaction_type") == "CREDIT", col("amount")).otherwise(0)).cast(DecimalType(18, 2)).alias("total_credit_amount"),
        _sum(when(col("transaction_type") == "DEBIT", col("amount")).otherwise(0)).cast(DecimalType(18, 2)).alias("total_debit_amount"),
        _max("transaction_timestamp").alias("last_transaction_date")
    )
)

# 3. Build Account Summary
account_summary_df = (
    dim_account
    .join(dim_customer.select("customer_id", "first_name", "last_name", "risk_category"), "customer_id", "left")
    .join(dim_branch.select("branch_id", "branch_name", "city", "state"), "branch_id", "left")
    .join(account_tx_agg, "account_id", "left")
    .select(
        dim_account["account_id"],
        dim_account["account_number"],
        dim_account["account_type"],
        dim_account["currency"],
        dim_account["balance"].cast(DecimalType(18, 2)).alias("current_balance"),
        dim_account["account_status"],
        dim_account["opened_date"],
        dim_customer["customer_id"],
        col("first_name"),
        col("last_name"),
        col("risk_category").alias("customer_risk_category"),
        dim_branch["branch_id"],
        col("branch_name"),
        col("city").alias("branch_city"),
        col("state").alias("branch_state"),
        coalesce(col("total_transactions"), lit(0)).alias("total_transactions"),
        coalesce(col("total_credit_amount"), lit(0.00)).cast(DecimalType(18, 2)).alias("total_credit_amount"),
        coalesce(col("total_debit_amount"), lit(0.00)).cast(DecimalType(18, 2)).alias("total_debit_amount"),
        col("last_transaction_date"),
        current_timestamp().alias("gold_ingestion_timestamp")
    )
)

# 4. Write to ADLS & Unity Catalog
account_summary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_PATH)
account_summary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FULL_TABLE_NAME)

print(f"Account Summary Gold Table Created: {account_summary_df.count()} records written to '{FULL_TABLE_NAME}'.")

Account Summary Gold Table Created: 200000 records written to '`banking_lakehouse_db2`.gold.account_summary'.
